# 04 — EDA: Hourly Patterns & Holidays

## Purpose
This notebook focuses on **time-of-day behavior** and **holiday effects** — two patterns
that are highly actionable for daily stock planning.

The hour-by-day-of-week heatmap is one of the most operationally useful outputs in the
entire project: it tells you exactly when each day's rush happens so stock can be ready at the right time.

## What we analyze
- Create holiday columns (Mexican federal holidays 2022–2026)
- Heatmap: how many orders arrive in each hour of each weekday
- Same heatmap broken down per branch and per temperature group
- Top day+hour combinations (peak moments)

## Input
`data/intermediate/datanomodifier.csv`

## Run order
Run after `01_data_cleaning.ipynb`.

## Setup — Load and clean data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

PROJECT_ROOT     = os.path.abspath(os.path.join(os.getcwd(), ".."))
INTERMEDIATE_DIR = os.path.join(PROJECT_ROOT, "data", "intermediate")

pd.set_option("display.max_columns", None)

df = pd.read_csv(os.path.join(INTERMEDIATE_DIR, "datanomodifier.csv"), low_memory=False)
for col in ["operating_date", "closing_time", "captured_time"]:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")
df["is_modifier"] = (df["is_modifier"].astype("string").str.strip().str.lower().map({"true": True, "false": False}).fillna(False).astype(bool))
df["quantity"] = pd.to_numeric(df["quantity"], errors="coerce").fillna(0)
df["tavg"] = pd.to_numeric(df["tavg"], errors="coerce")
df = df[~df["group"].isin(["CAFE Y BEBIDAS CALIENTES", "JUGOS Y BEBIDAS FRIAS"])].copy()
df["sucursal"] = df["sucursal"].replace({"Panem - Hotel Kavia N": "Panem - Hotel Kavia", "Panem - Plaza QIN N": "Panem - Plaza QIN", "Panem - Hospital Zambrano N": "Panem - Hospital Zambrano", "Panem - La Carreta N": "Panem - Carreta"})
df["item"] = df["item"].replace({"SANDWITCH ENSALADA POLLO": "SANDWICH ENSALADA POLLO"})
df["cold_or_warm"] = np.where(df["tavg"] >= 25, "warm", "cold")

# Extract hour and month from the capture timestamp
df["hour"] = df["captured_time"].dt.hour
df["month"] = df["captured_time"].dt.month

print(f"Rows loaded: {len(df):,}")

## Step 1 — Create holiday columns

We manually map Mexican federal holiday dates (`Festivo Oficial`) and bridge holidays
(`Puente` — long weekends created by Mexican labor law) for the full data range (2022–2026).

**Why do holidays matter for forecasting?**
On holidays, branches may be closed or serve a different customer mix (families vs office workers).
Holiday flag is one of the features in the ML model.

In [ ]:
# Map each holiday date to its type
holiday_map = {
    # 2022
    "2022-01-01": "Festivo Oficial", "2022-02-07": "Puente", "2022-03-21": "Puente",
    "2022-05-01": "Festivo Oficial", "2022-09-16": "Puente", "2022-11-21": "Puente", "2022-12-25": "Festivo Oficial",
    # 2023
    "2023-01-01": "Festivo Oficial", "2023-02-06": "Puente", "2023-03-20": "Puente",
    "2023-05-01": "Puente", "2023-09-16": "Festivo Oficial", "2023-11-20": "Puente", "2023-12-25": "Puente",
    # 2024
    "2024-01-01": "Puente", "2024-02-05": "Puente", "2024-03-18": "Puente",
    "2024-05-01": "Festivo Oficial", "2024-06-02": "Festivo Oficial", "2024-09-16": "Puente",
    "2024-10-01": "Festivo Oficial", "2024-11-18": "Puente", "2024-12-25": "Festivo Oficial",
    # 2025
    "2025-01-01": "Festivo Oficial", "2025-02-03": "Puente", "2025-03-17": "Puente",
    "2025-05-01": "Festivo Oficial", "2025-09-16": "Festivo Oficial", "2025-11-17": "Puente", "2025-12-25": "Festivo Oficial",
    # 2026
    "2026-01-01": "Festivo Oficial", "2026-02-02": "Puente", "2026-03-16": "Puente",
    "2026-05-01": "Puente", "2026-09-16": "Festivo Oficial", "2026-11-16": "Puente", "2026-12-25": "Puente",
}

holiday_map_dt = {pd.to_datetime(k): v for k, v in holiday_map.items()}

df["operating_date"] = pd.to_datetime(df["operating_date"], errors="coerce").dt.normalize()
df["holiday_type"] = df["operating_date"].map(holiday_map_dt).fillna("No holiday")
df["holidays"] = df["holiday_type"] != "No holiday"

print("Holiday type distribution:")
print(df["holiday_type"].value_counts())

## Analysis 1 — Orders per hour × day-of-week heatmap (all branches)

This is the most operationally useful chart in the project.

Each cell shows how many order lines were placed at a given hour on a given weekday,
accumulated across all data. Bright cells = peak demand moments.

**How to read it:**
- Rows = days of the week (Monday → Sunday)
- Columns = hours 0–23
- Color intensity = volume of orders
- High values at 8–10 AM = morning rush; at 12–14 = lunch rush

In [ ]:
day_order_map = {
    "lunes": 0, "martes": 1, "miércoles": 2, "miercoles": 2,
    "jueves": 3, "viernes": 4, "sábado": 5, "sabado": 5, "domingo": 6
}

table = df.pivot_table(index="day_name", columns="hour", aggfunc="size", fill_value=0)
table = table.sort_index(key=lambda idx: idx.str.lower().map(day_order_map))
table = table.reindex(columns=range(24), fill_value=0)
arr = table.values

fig, ax = plt.subplots(figsize=(16, 6))
im = ax.pcolormesh(arr, cmap="YlOrRd", vmin=0)
plt.colorbar(im, ax=ax, label="Order count")

ax.set_yticks(np.arange(len(table.index)) + 0.5)
ax.set_yticklabels(table.index, rotation=0)
ax.set_xticks(np.arange(24) + 0.5)
ax.set_xticklabels(range(24))
ax.set_xlim(0, 24)

# Annotate each cell with its value
mx = arr.max() if arr.max() > 0 else 1
for i in range(arr.shape[0]):
    for j in range(arr.shape[1]):
        val = int(arr[i, j])
        color = "white" if val > mx * 0.65 else "black"
        ax.text(j + 0.5, i + 0.5, str(val), ha="center", va="center", fontsize=7, color=color)

ax.set_xlabel("Hour of day")
ax.set_ylabel("Day of week")
ax.set_title("Orders per Hour × Day of Week (All Branches Combined)")
plt.tight_layout()
plt.show()

## Analysis 2 — Top day + hour combinations

A ranked table of the highest-volume (day, hour) pairs.
This is the peak demand summary — the moments when you absolutely need to have stock ready.

In [ ]:
top_day_hour = (
    df.groupby(["day_name", "hour"])
      .size()
      .reset_index(name="order_count")
      .sort_values("order_count", ascending=False)
)
top_day_hour.index = range(1, len(top_day_hour) + 1)
top_day_hour.head(15)

## Analysis 3 — Hour × day heatmap per branch and temperature group

We now split the heatmap by **branch** and by **cold_or_warm**.

This reveals whether rush hours shift on warm vs cold days, which could affect
how early you need to prepare cold items (salads, cold dishes) vs warm items (soups, hot pastries).

In [ ]:
for branch in sorted(df["sucursal"].dropna().unique()):
    for temp in ["cold", "warm"]:
        sub = df[(df["sucursal"] == branch) & (df["cold_or_warm"] == temp)].copy()
        if sub.empty:
            continue

        t = sub.pivot_table(index="day_name", columns="hour", aggfunc="size", fill_value=0)
        t = t.sort_index(key=lambda idx: idx.str.lower().map(day_order_map))
        t = t.reindex(columns=range(24), fill_value=0)
        arr = t.values

        fig, ax = plt.subplots(figsize=(16, 5))
        im = ax.pcolormesh(arr, cmap="YlOrRd", vmin=0)
        plt.colorbar(im, ax=ax, label="Count")
        ax.set_yticks(np.arange(len(t.index)) + 0.5)
        ax.set_yticklabels(t.index)
        ax.set_xticks(np.arange(24) + 0.5)
        ax.set_xticklabels(range(24))
        ax.set_xlim(0, 24)

        mx = arr.max() if arr.max() > 0 else 1
        for i in range(arr.shape[0]):
            for j in range(arr.shape[1]):
                val = int(arr[i, j])
                color = "white" if val > mx * 0.65 else "black"
                ax.text(j + 0.5, i + 0.5, str(val), ha="center", va="center", fontsize=7, color=color)

        ax.set_title(f"{branch} | {temp.upper()} days — Orders per Hour × Day")
        ax.set_xlabel("Hour")
        plt.tight_layout()
        plt.show()

## Summary

Key takeaways:
- There are clear peak hours on every weekday — morning rush and lunch rush are the dominant patterns
- Peaks may shift slightly between cold and warm days
- Holiday counts are relatively small but can create visible dips in demand

**Next step:** `05_visualizations.ipynb` — interactive Sankey diagrams and trend line charts.